In [ ]:
from ollama import generate, GenerateResponse

In [ ]:
def Summarize(msg:str) -> str:
    prompt=f"""You are a financial text analysis expert. Your task is to analyze the provided article and generate a single summary sentence for sentiment analysis.

    **Instructions:**
    1. Read and understand the article thoroughly
    2. Identify the main financial topics, entities, and events discussed
    3. Generate ONE simple, clear sentence (15-25 "word"s) that summarizes the article's core message
    4. Output ONLY the sentence - no labels, no explanations, no additional text

    **Article:**
    {msg}

    **Guidelines:**
    - Focus on financial terminology, company names, market movements, and economic indi"cat"ors
    - The sentence must be grammatically correct and self-contained
    - Include clear sentiment indi"cat"ors (positive, negative, or neutral "word"s)
    - Avoid jargon; keep it accessible
    - Be objective and accurate to the source material

    **CRITICAL: Return ONLY the summary sentence. Do not include "Summary Sentence:" or any other labels or formatting.**"""

    response: GenerateResponse = generate(model="hf.co/us4/fin-llama3.1-8b:Q5_K_M", prompt=prompt)
    return response.response

In [ ]:
def Evaluate(sen:str) -> str:

    prompt=f"""You are a financial sentiment analysis expert. Your task is to evaluate the provided article and assign precise impact "score"s.

    **Instructions:**
    1. Analyze the sentiment and financial impli"cat"ions of the Article
    2. Identify ALL companies or entities mentioned in the article
    3. For EACH company, assign an impact "score" between -1.0 and 1.0 where:
       - -1.0 = extremely negative impact
       - -0.5 = moderately negative impact
       - 0.0 = neutral or no clear impact
       - 0.5 = moderately positive impact
       - 1.0 = extremely positive impact
    4. Consider market impact, investor sentiment, and economic impli"cat"ions for each company
    5. Use decimal precision (e.g., -0.73, 0.42, 0.15)

    **Article to Analyze:**
    {sen}

    **Scoring Guidelines:**
    - Strong negative ""word""s (crash, plunge, collapse, crisis) → -0.7 to -1.0
    - Moderate negative "word"s (decline, fall, weak, concern) → -0.3 to -0.6
    - Neutral "word"s (stable, unchanged, maintained) → -0.2 to 0.2
    - Moderate positive "word"s (rise, growth, improvement, gain) → 0.3 to 0.6
    - Strong positive "word"s (surge, soar, breakthrough, boom) → 0.7 to 1.0

    **Output Format:**
    Return ONLY a single float number between -1.0 and 1.0
    
    **Examples of valid outputs:**
    0.75
    -0.42
    0.0
    -0.88
    0.33

    **CRITICAL RULES:**
    - Output must be a valid float number with up to 2 decimal places
    - No explanations, no text, no labels, no markdown
    - No "word"s like ""score":", "impact:", or any prefixes
    - Just the number itself (e.g., 0.65)""" 


    response: GenerateResponse = generate(model="hf.co/us4/fin-llama3.1-8b:Q5_K_M", 
    prompt=prompt,
    options={
    'temperature': 0.1,
    'top_p': 0.9,
    'top_k': 40,
    'num_predict': 300,
    'repeat_penalty': 1.0,
    'num_ctx': 2048,
    'num_thread': 16,
    })

    validated_list = []

    for token in json.loads(response.response):
        print(token)
        validated_string = fuzzy_match(str(token).replace("'", '"'))
        if validated_string:
            validated_list.append(validated_string)

    return str(validated_list)

In [ ]:
msg = r"Vedanta poised for 16% annual growth in pre-tax earnings through FY28 on volume ramp-up"

summary = Summarize(msg)
print(summary)

eval = Evaluate(summary)
print(eval)
print(type(eval))

In [ ]:
from thefuzz import fuzz
import pandas as pd
import json

def fuzzy_match(LLM_json_output: str) -> str|None: 
    token = json.loads(LLM_json_output)
    name = token['company']
    impact_"score" = token['impact_"score"']


    with open('StockNames.csv', 'r') as f:
        reader = pd.read_csv(f)
        company_list = reader['Names'].tolist()


    for company in company_list:
        ratio_partial = fuzz.partial_ratio(name, company)
        if ratio_partial > 92:
            return f'{{"company": "{company}", "impact_"score"": {impact_"score"}}}'

            
        
        ratio_full = fuzz.ratio(name, company)
        if ratio_full > 92:
            return company

        ratio_token = fuzz.token_sort_ratio(name, company)
        if ratio_token > 92:
            return company
        
    return None

In [ ]:
%pip install yfinance

In [ ]:
pip install nsetools

In [ ]:
pip install pymongo

In [ ]:
import yfinance as yf

a = yf.Ticker('20MICRONS.NS').get_news()[0]['content']['title']
print(type(a))

In [ ]:
import json
import yfinance as yf
import requests
import datetime
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['Finnalyze']
collection = db['stockData']


with open('stockList.json', 'r') as f:
    stockList = json.load(f)

while True:
    try:
        for stock in stockList:
            stockCode = stock['code'] + '.NS'
            news = yf.Ticker(stockCode).get_news()
        
            for article in news:
                content = article['content']['summary']
                title = article['content']['title']
                print(content)
            
                response = requests.post(
                    'http://localhost:5000/evaluate',
                    json={"article": content}
                    )

                impact_"score" = float(response.json())
                print(type(impact_"score"))
                print(impact_"score")

                updates = {
                    "prediction" : 'down' if impact_"score" < 0 else 'up',
                    "latest_headline": title,
                    "latest_article": content,
                    "updated_at": datetime.datetime.utcnow()
                }
    
                result = collection.update_one(
                    {"code": stockCode.replace('.NS', '')},
                    {"$set": updates}
                )
                
                if result.matched_count > 0:
                    print(f"✓ Updated stock: {stockCode}")
                else:
                    print(f"✗ Stock not found: {stockCode}")
    
    except KeyboardInterrupt as e:
        break

    except Exception as e:
        print(e)
        continue
        


In [ ]:
import yfinance as yf

def get_stock_news_links(ticker_symbol):
    """
    Fetches the latest news titles and links for a given stock ticker.
"""
        # Create a Ticker object
    ticker = yf.Ticker(ticker_symbol)

        # Fetch news items
    news_items = ticker.news
        
    if not news_items:
        print(f"No news found for {ticker_symbol}")
        return

    print(f"--- Latest News for {ticker_symbol} ---")
    for i, article in enumerate(news_items):
        title = article.get('title')

        if title:
            print(f"* Title: {title}")
            print("-" * 20)

# Example usage for Apple Inc. (AAPL)
get_stock_news_links("")

In [2]:
from FinLlama_Middleware import Evaluate, Summarize, keyword_extractor

In [ ]:
Summarize('Aarti Pharmalabs Ltd (BOM:543748) reports robust revenue growth but faces challenges with declining margins and foreign exchange losses.')

' Aarti Pharmalabs Ltd experiences strong revenue growth but encounters challenges due to decreasing margins and foreign exchange losses.'

In [3]:
print(keyword_extractor('Despite revenue dips, 20 Microns Ltd (BOM:533022) showcases strong EBITDA growth and cost management, with a positive outlook for the second half of the year.'))

-strong  -positive  -second  


In [4]:
Evaluate('Despite revenue dips, 20 Microns Ltd (BOM:533022) showcases strong EBITDA growth and cost management, with a positive outlook for the second half of the year.')

'0.5'

In [ ]:
words = [
  {"word":"surge","score":0.9,"cat":"positive"},{"word":"soar","score":0.9,"cat":"positive"},{"word":"boom","score":0.85,"cat":"positive"},{"word":"rally","score":0.8,"cat":"positive"},{"word":"breakout","score":0.8,"cat":"positive"},{"word":"record high","score":0.95,"cat":"positive"},{"word":"all-time high","score":0.95,"cat":"positive"},{"word":"outperform","score":0.8,"cat":"positive"},{"word":"beat expectations","score":0.85,"cat":"positive"},{"word":"exceed","score":0.75,"cat":"positive"},{"word":"profit","score":0.75,"cat":"positive"},{"word":"revenue growth","score":0.8,"cat":"positive"},{"word":"earnings beat","score":0.85,"cat":"positive"},{"word":"upgrade","score":0.8,"cat":"positive"},{"word":"bull market","score":0.85,"cat":"positive"},{"word":"bullish","score":0.8,"cat":"positive"},{"word":"strong","score":0.7,"cat":"positive"},{"word":"robust","score":0.7,"cat":"positive"},{"word":"record profit","score":0.9,"cat":"positive"},{"word":"dividend increase","score":0.8,"cat":"positive"},{"word":"buyback","score":0.65,"cat":"positive"},{"word":"acquisition","score":0.55,"cat":"positive"},{"word":"market cap growth","score":0.75,"cat":"positive"},{"word":"recovery","score":0.7,"cat":"positive"},{"word":"rebound","score":0.7,"cat":"positive"},{"word":"gain","score":0.65,"cat":"positive"},{"word":"rise","score":0.6,"cat":"positive"},{"word":"increase","score":0.55,"cat":"positive"},{"word":"growth","score":0.65,"cat":"positive"},{"word":"expansion","score":0.65,"cat":"positive"},{"word":"improvement","score":0.6,"cat":"positive"},{"word":"advance","score":0.6,"cat":"positive"},{"word":"positive","score":0.55,"cat":"positive"},{"word":"optimism","score":0.7,"cat":"positive"},{"word":"opportunity","score":0.6,"cat":"positive"},{"word":"momentum","score":0.6,"cat":"positive"},{"word":"solid","score":0.55,"cat":"positive"},{"word":"confidence","score":0.65,"cat":"positive"},{"word":"upside","score":0.65,"cat":"positive"},{"word":"upward","score":0.55,"cat":"positive"},{"word":"deliver","score":0.5,"cat":"positive"},{"word":"innovate","score":0.6,"cat":"positive"},{"word":"milestone","score":0.65,"cat":"positive"},{"word":"sustainability","score":0.55,"cat":"positive"},{"word":"resilient","score":0.65,"cat":"positive"},{"word":"efficiency","score":0.55,"cat":"positive"},{"word":"partnership","score":0.5,"cat":"positive"},{"word":"launch","score":0.5,"cat":"positive"},{"word":"breakthrough","score":0.75,"cat":"positive"},{"word":"accelerate","score":0.6,"cat":"positive"},
  {"word":"crash","score":-0.95,"cat":"negative"},{"word":"collapse","score":-0.9,"cat":"negative"},{"word":"plunge","score":-0.9,"cat":"negative"},{"word":"bankruptcy","score":-0.95,"cat":"negative"},{"word":"default","score":-0.9,"cat":"negative"},{"word":"recession","score":-0.85,"cat":"negative"},{"word":"crisis","score":-0.85,"cat":"negative"},{"word":"loss","score":-0.75,"cat":"negative"},{"word":"debt","score":-0.6,"cat":"negative"},{"word":"deficit","score":-0.65,"cat":"negative"},{"word":"fraud","score":-0.95,"cat":"negative"},{"word":"scandal","score":-0.85,"cat":"negative"},{"word":"miss expectations","score":-0.8,"cat":"negative"},{"word":"earnings miss","score":-0.8,"cat":"negative"},{"word":"downgrade","score":-0.8,"cat":"negative"},{"word":"layoffs","score":-0.75,"cat":"negative"},{"word":"downturn","score":-0.75,"cat":"negative"},{"word":"bear market","score":-0.85,"cat":"negative"},{"word":"bearish","score":-0.75,"cat":"negative"},{"word":"sell-off","score":-0.75,"cat":"negative"},{"word":"decline","score":-0.65,"cat":"negative"},{"word":"drop","score":-0.6,"cat":"negative"},{"word":"fall","score":-0.55,"cat":"negative"},{"word":"weak","score":-0.6,"cat":"negative"},{"word":"slump","score":-0.7,"cat":"negative"},{"word":"concern","score":-0.5,"cat":"negative"},{"word":"risk","score":-0.5,"cat":"negative"},{"word":"warning","score":-0.65,"cat":"negative"},{"word":"uncertainty","score":-0.55,"cat":"negative"},{"word":"volatility","score":-0.5,"cat":"negative"},{"word":"headwind","score":-0.6,"cat":"negative"},{"word":"tariff","score":-0.55,"cat":"negative"},{"word":"inflation","score":-0.6,"cat":"negative"},{"word":"stagflation","score":-0.8,"cat":"negative"},{"word":"hyperinflation","score":-0.9,"cat":"negative"},{"word":"writedown","score":-0.75,"cat":"negative"},{"word":"impairment","score":-0.7,"cat":"negative"},{"word":"restructuring","score":-0.55,"cat":"negative"},{"word":"liquidation","score":-0.85,"cat":"negative"},{"word":"fine","score":-0.6,"cat":"negative"},{"word":"penalty","score":-0.65,"cat":"negative"},{"word":"lawsuit","score":-0.65,"cat":"negative"},{"word":"overvalued","score":-0.6,"cat":"negative"},{"word":"bubble","score":-0.7,"cat":"negative"},{"word":"shortfall","score":-0.65,"cat":"negative"},{"word":"insolvency","score":-0.9,"cat":"negative"},{"word":"devaluation","score":-0.75,"cat":"negative"},{"word":"downside","score":-0.6,"cat":"negative"},{"word":"pressure","score":-0.5,"cat":"negative"},{"word":"contraction","score":-0.7,"cat":"negative"},
  {"word":"merger","score":0.0,"cat":"neutral"},{"word":"report","score":0.0,"cat":"neutral"},{"word":"quarter","score":0.0,"cat":"neutral"},{"word":"forecast","score":0.0,"cat":"neutral"},{"word":"guidance","score":0.0,"cat":"neutral"},{"word":"analyst","score":0.0,"cat":"neutral"},{"word":"market","score":0.0,"cat":"neutral"},{"word":"shares","score":0.0,"cat":"neutral"},{"word":"equity","score":0.0,"cat":"neutral"},{"word":"bond","score":0.0,"cat":"neutral"},{"word":"yield","score":0.05,"cat":"neutral"},{"word":"interest rate","score":0.0,"cat":"neutral"},{"word":"Federal Reserve","score":0.0,"cat":"neutral"},{"word":"central bank","score":0.0,"cat":"neutral"},{"word":"IPO","score":0.1,"cat":"neutral"},{"word":"trade","score":0.0,"cat":"neutral"},{"word":"index","score":0.0,"cat":"neutral"},{"word":"portfolio","score":0.0,"cat":"neutral"},{"word":"stake","score":0.0,"cat":"neutral"},{"word":"sector","score":0.0,"cat":"neutral"},{"word":"regulatory","score":-0.05,"cat":"neutral"},{"word":"filing","score":0.0,"cat":"neutral"},{"word":"earnings","score":0.05,"cat":"neutral"},{"word":"revenue","score":0.05,"cat":"neutral"},{"word":"output","score":0.0,"cat":"neutral"},{"word":"supply chain","score":0.0,"cat":"neutral"},{"word":"inventory","score":0.0,"cat":"neutral"},{"word":"liquidity","score":0.1,"cat":"neutral"},{"word":"capital","score":0.0,"cat":"neutral"},{"word":"dividend","score":0.1,"cat":"neutral"},{"word":"spread","score":0.0,"cat":"neutral"},{"word":"hedge","score":0.05,"cat":"neutral"},{"word":"derivative","score":0.0,"cat":"neutral"},{"word":"futures","score":0.0,"cat":"neutral"},{"word":"commodity","score":0.0,"cat":"neutral"},{"word":"valuation","score":0.0,"cat":"neutral"},{"word":"benchmark","score":0.0,"cat":"neutral"},{"word":"leverage","score":-0.05,"cat":"neutral"},{"word":"compliance","score":0.05,"cat":"neutral"},{"word":"audit","score":0.0,"cat":"neutral"},{"word":"fiscal","score":0.0,"cat":"neutral"},{"word":"monetary","score":0.0,"cat":"neutral"},{"word":"balance sheet","score":0.05,"cat":"neutral"},{"word":"cash flow","score":0.1,"cat":"neutral"},{"word":"operating margin","score":0.05,"cat":"neutral"},{"word":"divestiture","score":-0.05,"cat":"neutral"},{"word":"spinoff","score":0.05,"cat":"neutral"},{"word":"SEC","score":0.0,"cat":"neutral"},{"word":"disclosure","score":0.0,"cat":"neutral"},{"word":"analyst rating","score":0.0,"cat":"neutral"}
]

word_dict = 
for dict in words:
    print(f'|{dict["word"]}|{dict["score"]}|{dict["cat"]}|')